In [36]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [37]:
import os

os.getcwd()

'/home/onyxia/work/stat-ut-avengers'

In [38]:
os.chdir("/home/onyxia/work/stat-ut-avengers")

In [39]:
from pypdf import PdfReader

reader2024 = PdfReader("docs/aarsregnskap_928434885-2024.pdf")
reader2023 = PdfReader("docs/aarsregnskap_928434885-2023.pdf")
print(len(reader2024.pages))

24


In [40]:
import pandas as pd
import random
import numpy as np

# Poster hentet fra vedlegget
poster = [
    "03100", "03200", "74900", "80300", "80500",
    "80051", "80052", "80900", "81300", "81500",
    "81701", "20000", "20001", "20002", "20003",
    "20004", "20100", "20200", "20300", "20410",
    "20420", "20450", "20550", "20590", "20800",
    "13120", "13130", "13200", "13310", "13320",
    "13400", "13500", "13600", "13700", "13800",
    "13900", "15000", "15010", "15300", "15600",
    "15650", "15700", "17800", "18000", "18100",
    "18300", "18400", "18800", "18900", "19200",
    "22000", "22100", "22200", "22500", "22600",
    "22800", "22900", "23100", "23200", "23300",
    "23800", "24000", "24600", "28000", "29000",
    "29100", "29200", "29490", "29500", "29700",
    "29900"
]

valutaer = ["NOK", "EUR", "USD", "SEK", "DKK", "GBP"]
land = ["NO", "SE", "DK", "FI", "DE", "FR", "US", "GB"]
perioder = ["202103", "202106", "202309", "202312"]

rows = []

for _ in range(2000):

    post = random.choice(poster)

    # Underpost: mest 00, sjelden 99
    under_post = random.choices(
        ["00", "99"],
        weights=[0.95, 0.05]
    )[0]

    # 9-sifret orgnummer
    org_nummer = str(random.randint(100000000, 999999999))

    # Verdi mellom 0 og 100 millioner
    verdi = round(random.uniform(0, 100_000_000), 2)

    periode = random.choice(perioder)

    landkode = random.choice(land)

    valuta = random.choice(valutaer)

    # Prefix = to tall
    prefix = str(random.randint(10, 99))

    rows.append({
        "post": post,
        "under_post": under_post,
        "org_nummer": org_nummer,
        "verdi": str(verdi),
        "periode": periode,
        "land": landkode,
        "valuta": valuta,
        "prefix": prefix
    })

df = pd.DataFrame(rows)

# Alle kolonner som object/string
df = df.astype(str)

print(df.head())

# Lagre til CSV
df.to_csv("syntetiske_data.csv", index=False)

    post under_post org_nummer        verdi periode land valuta prefix
0  29500         00  359535929  17154536.93  202103   NO    SEK     15
1  18800         00  562131301  81678489.57  202309   NO    USD     48
2  19200         99  573802692  69357861.19  202312   FR    EUR     95
3  13200         00  778643040  86823329.18  202106   DK    EUR     61
4  20800         00  895350689  34070605.44  202103   GB    NOK     32


In [41]:
df.dtypes

post          object
under_post    object
org_nummer    object
verdi         object
periode       object
land          object
valuta        object
prefix        object
dtype: object

In [42]:
df["periode"] = df["periode"].astype(str).str.strip()
df["verdi"] = pd.to_numeric(df["verdi"], errors="coerce").fillna(0)

df = (
    df.groupby(["org_nummer", "post"])
      .apply(
          lambda g:
          g.loc[g["periode"] == "202312", "verdi"].sum()
          -
          g.loc[g["periode"] == "202103", "verdi"].sum()
      )
      .reset_index(name="diff")
)

/tmp/ipykernel_111380/1802588668.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [43]:
df["diff"].value_counts()

diff
 0.00           1037
 6418584.63        1
-92153642.63       1
-24118069.07       1
 72971517.68       1
                ... 
 89311538.26       1
-26003243.73       1
-32780699.61       1
-47835063.57       1
-27257730.10       1
Name: count, Length: 964, dtype: int64

In [44]:
X=df.drop(columns=["diff"])
y=df["diff"]

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
#from preprocess import complete_pre_processing
#from log_mlflow import log_to_mlflow
#from pipeline import set_pipeline
#from utils import setup_logging, set_seed, check_data, store_datasets, store_model_mlflow_s3
import warnings

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)